In [1]:
%load_ext autoreload
%autoreload 2
import torch
import numpy as np
import os
import matplotlib.pyplot as plt
from dataset import BanditTaskNeuroPixelsDataset
from ncmcm.bundlenet.bundlenet import BunDLeNet, train_model, project_into_latent_space
from ncmcm.bundlenet.utils import prep_data, timeseries_train_test_split
from ncmcm.visualisers.neuronal_behavioural import plotting_neuronal_behavioural
from ncmcm.visualisers.latent_space import LatentSpaceVisualiser

/home/kerim/Projects/Neural Algorithms/NC-MCM/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-18 13:18:29,479	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


## Dataset Overview

We apply **BunDLe-Net** to neural recordings from mice performing a two-armed bandit task. In this task, the animal must learn to identify which of two actions yields the highest reward through trial and error. The neural data captures the decision-making process across multiple trials as the animal learns the optimal policy.

The dataset includes Neuropixels recordings with high-dimensional neural activity aligned to behavioral choices (left or right arm selection) and trial outcomes (reward or no reward).

## Loading the Data

Load the neural activity (`x`) and behavioral choice (`b`) data. We use the `BanditTaskNeuroPixelsDataset` class which handles the two-armed bandit task data.

In [2]:
# Load the 2-arm bandit task dataset
dataset_path = os.path.abspath("/home/kerim/Projects/Neural Algorithms/NC-MCM/datasets/raw/twoArmBandit/JPAS_0023_20230922")
dataset = BanditTaskNeuroPixelsDataset(data_path=dataset_path, downsample_fs=10, downsample_method='count', good_neurons_only=True)
x = dataset.x.T.toarray()  # shape: (timepoints, n_neurons)
b = dataset.b.toarray().flatten()  # shape: (timepoints) 
b_labels = dataset.b_labels  # mapping of behavior indices to names

Loaded BanditTaskNeuroPixelsDataset from /home/kerim/Projects/Neural Algorithms/NC-MCM/datasets/raw/twoArmBandit/JPAS_0023_20230922
Neuronal data shape: (367, 12931), Behavioral data shape: (1, 12931), Sampling frequency: 10 Hz
Behavioral labels: {0: 'intertrial', 1: 'hold', 2: 'reward', 3: 'no reward', 4: 'choosing left', 5: 'choosing right'}


In [ ]:
b_labels

In [ ]:
# Visualize the neural activity and behavioral choices
plotting_neuronal_behavioural(x, b, b_names=b_labels)

## Preparing Data for BunDLe-Net  

Ensure that your discrete behavioral labels are consecutive integers from `0` to `n_states -1`. For the 2-arm bandit task, the choices are already binary (0 and 1).

In [ ]:
# For this dataset, choices are already in the correct format (integers starting from 0)
# If needed, use LabelEncoder for other datasets:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
b = label_encoder.fit_transform(b)

The **`prep_data`** function transforms the time-series neural activity (`x`) and behavioral (`b`) data into a format suitable for the **BunDLe-Net** model. The **`win`** parameter specifies the window length, and allows us to perform a time delay embedding:
- **`win=1`**: Embeds a single time step.  
- **`win>1`**: Embeds a window of time-slices.

#### Output Shapes  
- **Neural data (`x_`)**: Shaped as **(t - win, 2, win, n)**, where:  
  - `x_[:,0,:,:]` represents $X_t$  
  - `x_[:,1,:,:]` represents $X_{t+1}$
  - the first dim denotes time-steps
  - the last dim denotes feature number (number of neurons)
- **Behavioral data (`b_`)**: Shaped as **(t - win, ...)** to maintain synchronization with `x_`.  

For this dataset, we use `prep_data` with `win=1`:  

In [ ]:
print(x.shape, b.shape)
x_, b_ = prep_data(x, b, win=1)
print(x_.shape, b_.shape)

In [ ]:
x_train, x_test, b_train_1, b_test_1 = timeseries_train_test_split(x_, b_)

In [ ]:
print(x_train.shape, b_train_1.shape)
print(x_test.shape, b_test_1.shape)

## Training BunDLe-Net

Now, train BunDLe-Net on the prepared neural activity and behavioral choice data from the 2-arm bandit task.

In [ ]:
# (re)create model and ensure it's on CPU
num_behaviour = len(np.unique(b))
model = BunDLeNet(latent_dim=3, num_behaviour=num_behaviour, input_shape=x_train.shape)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Train on CPU. If train_model accepts a device argument, pass it; otherwise call without it.
loss_array_train, loss_array_test = train_model(
    x_train,
    b_train_1,
    model,
    b_type='discrete',
    gamma=0.9,
    learning_rate=0.001,
    n_epochs=350,
    batch_size=200,
    device=device, 
    validation_data=(x_test, b_test_1)
)


## Training loss versus epochs

We plot the losses over the epochs in order to check whether the model has converged during training.

In [ ]:
for i, label in enumerate([
    r"$\mathcal{L}_{\mathrm{Markov}}$",
    r"$\mathcal{L}_{\mathrm{Behavior}}$",
    r"Total loss $\mathcal{L}$"
]):
    plt.semilogy(loss_array_train[:, i], label=label)
plt.legend()
plt.show()

In [ ]:
for i, label in enumerate([
    r"$\mathcal{L}_{\mathrm{Markov}}$",
    r"$\mathcal{L}_{\mathrm{Behavior}}$",
    r"Total loss $\mathcal{L}$"
]):
    plt.semilogy(loss_array_test[:, i], label=label)
plt.legend()
plt.show()

## Projecting into latent space

Once we confirm the model has converged, we project the traces into the learned latent space.

In [ ]:
y0_train = project_into_latent_space(x_train, model)

In [ ]:
y0_test = project_into_latent_space(x_test, model)

## Visualising embedding
Using the LatentSpaceVisualiser class, we first visualise the embedding as time series to see how the embedding corresponds to behavior.

In [ ]:
# Plotting latent space dynamics
vis = LatentSpaceVisualiser(y0_train, b_train_1, b_labels, show_points=True)
vis.plot_latent_timeseries(filename='./figures/latent_time_series_bundlenet_2arm_bandit.png')

We then use the LatentSpaceVisualiser class to explore the temporal dynamics in phase space (state space). This class allows us to plot arrows between time points at $t$ and $t+1$, helping us identify temporal structures and decision-related dynamics.

In [ ]:
%matplotlib inline
vis.plot_phase_space(filename='./figures/phase_space_dynamics_bundlenet_2arm_bandit.png')

In [ ]:
### Run to produce rotating 3-D plot
# %matplotlib inline
# vis.rotating_plot(filename='./figures/rotation_bundlenet_2arm_bandit.gif')

In [ ]:
### Run to produce interactive 3-D plot 
# %matplotlib inline
_ = vis.plot_interactive_3d(show_fig=True, filename='./figures/interactive_3d_bundlenet_2arm_bandit.html')

### Recurrence plot analysis of BunDLeNet's embedding

In [ ]:
plt.close('all')

In [ ]:
%matplotlib inline
pd_y = np.linalg.norm(y0_train[:1000, np.newaxis] - y0_train[:1000], axis=-1) < 0.8
plt.figure()
plt.matshow(pd_y, cmap='Greys')
plt.show()

# Load latent trajectory and visualize

Load already computed latent trajectory and visualize it.